In [8]:
# Notebook 4: Embeddings & Qdrant Vector DB (200k labeled conversations)

!pip install -q sentence-transformers qdrant-client torch pandas tqdm

import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.auto import tqdm
from pathlib import Path
import os

from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = Path('/content/drive/MyDrive/AIE_project/outputs')
CONV_PATH = OUT_DIR / 'conversations_labeled.csv'   # from Notebook 2 (200k rows)
QDRANT_PATH = '/tmp/qdrant_storage'
os.makedirs(QDRANT_PATH, exist_ok=True)

df = pd.read_csv(CONV_PATH)
print(f'Loaded {len(df):,} labeled conversations (200k sample)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 200,000 labeled conversations (200k sample)


In [9]:
# Optional: keep only tweets that have a company reply (they all do because we sampled? Not necessarily; let's filter)
df = df[df['company_reply'].notna()].reset_index(drop=True)
print(f'After filtering to tweets with reply: {len(df):,} rows')

After filtering to tweets with reply: 152,509 rows


In [10]:
# Sample for embedding (you can adjust or use all)
# Since we already have 200k, we can use all of them, or sample further if needed.
USE_ALL = True
if not USE_ALL:
    SAMPLE_SIZE = 100_000
    df = df.groupby('priority', group_keys=False).apply(
        lambda x: x.sample(min(len(x), SAMPLE_SIZE // 2), random_state=42)
    ).reset_index(drop=True)
    print(f'Sampled to {len(df):,} for embedding')

In [11]:
# Embedding model on GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
EMBED_DIM = 384

texts_to_embed = df['conversation'].fillna(df['text_clean']).tolist()
print(f'Number of texts to embed: {len(texts_to_embed):,}')

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of texts to embed: 152,509


In [12]:
import shutil
import os

QDRANT_PATH = '/tmp/qdrant_storage'

# Delete the entire Qdrant storage folder to remove any stale lock
if os.path.exists(QDRANT_PATH):
    shutil.rmtree(QDRANT_PATH)
    print(f'Removed existing Qdrant storage: {QDRANT_PATH}')

# Create fresh folder
os.makedirs(QDRANT_PATH, exist_ok=True)

# Initialize Qdrant client
qdrant = QdrantClient(path=QDRANT_PATH)
COLL_NAME = 'support_tickets'

# Create collection (no need to check for existing, folder is new)
qdrant.create_collection(
    collection_name=COLL_NAME,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE)
)
print(f'Collection {COLL_NAME} created at {QDRANT_PATH}')

Removed existing Qdrant storage: /tmp/qdrant_storage
Collection support_tickets created at /tmp/qdrant_storage


In [13]:
EMBED_BATCH_SIZE = 64
UPSERT_BATCH_SIZE = 256
total_stored = 0

# Pre-embed everything at once if memory allows
all_embeddings = embedder.encode(
    texts_to_embed,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    device='cuda'  # explicit GPU
)

# Then upsert in batches
for start in tqdm(range(0, len(df), UPSERT_BATCH_SIZE)):
    batch_df = df.iloc[start:start+UPSERT_BATCH_SIZE]
    batch_embs = all_embeddings[start:start+UPSERT_BATCH_SIZE]

    points = [
        PointStruct(
            id=total_stored + i,
            vector=emb.tolist(),
            payload={
                'tweet_id': str(row.tweet_id),
                'customer_text': str(row.text_clean)[:500],
                'company_reply': str(row.company_reply)[:500] if pd.notna(row.company_reply) else None,
                'conversation': str(row.conversation)[:800],
                'priority': int(row.priority),
                'priority_label': 'urgent' if row.priority == 1 else 'normal'
            }
        )
        for i, (emb, row) in enumerate(zip(batch_embs, batch_df.itertuples()))
    ]

    qdrant.upsert(collection_name=COLL_NAME, points=points)
    total_stored += len(points)

print(f'✅ Stored {total_stored:,} vectors')

Batches:   0%|          | 0/2383 [00:00<?, ?it/s]

  0%|          | 0/596 [00:00<?, ?it/s]

✅ Stored 152,509 vectors


In [14]:
# ✅ Copy Qdrant DB from local /tmp to Drive (runs once, sequential = fast)
import shutil

DRIVE_QDRANT = str(OUT_DIR / 'qdrant_db')

print("Copying Qdrant storage to Drive...")
if os.path.exists(DRIVE_QDRANT):
    shutil.rmtree(DRIVE_QDRANT)          # remove old version

shutil.copytree(QDRANT_PATH, DRIVE_QDRANT)
print(f"✅ Saved to Drive: {DRIVE_QDRANT}")

Copying Qdrant storage to Drive...
✅ Saved to Drive: /content/drive/MyDrive/AIE_project/outputs/qdrant_db


In [16]:
# Quick test
test_queries = [
    'my package never arrived and no one is responding',
    'you charged me twice for the same order',
]
print('=== RETRIEVAL TEST ===')
for query in test_queries:
    print(f'\nQuery: "{query}"')
    query_emb = embedder.encode(query, normalize_embeddings=True).tolist()
    response = qdrant.query_points(
        collection_name=COLL_NAME,
        query=query_emb,
        limit=2,
        with_payload=True
    )
    for r in response.points:
        print(f'  Score: {r.score:.4f} | Priority: {r.payload["priority_label"]}')
        print(f'  Customer: {r.payload["customer_text"][:80]}...')

=== RETRIEVAL TEST ===

Query: "my package never arrived and no one is responding"
  Score: 0.7176 | Priority: normal
  Customer: Hi my package was delivered yesterday but I never received the package. I was ho...
  Score: 0.7041 | Priority: normal
  Customer: I never received a package I ordered. It says it was delivered to my mailbox and...

Query: "you charged me twice for the same order"
  Score: 0.7711 | Priority: normal
  Customer: was just charged twice for the same order. How do I resolve??...
  Score: 0.7656 | Priority: normal
  Customer: who can I speak to about being charge twice for an order?...
